# Agentic-Fuzzing -- Kaggle Notebook (Open-Source LLM Edition)

Runs the **full Agentic-Fuzzing** pipeline on Kaggle using an **open-source model**
(HuggingFace Hub **or** a Kaggle Model) instead of the Groq API:

> LLM (open-source) -> Hypothesis XML strategy -> C harness (ASan/UBSan) -> crash triage

| Pipeline stage | What happens |
|---|---|
| LLM (open-source) | Writes a `hypothesis` XML-generating strategy |
| `generator/` | AST-validates the strategy, then live-loads it |
| C harness `mxml_harness` | Parses each XML under ASan/UBSan, classifies exit code |
| Refine loop | Feeds acceptance-rate + crash signatures back to the LLM |
| `triage/` | Deduplicates, minimizes & verifies any crashes found |

### Before you run (important)
1. **Settings -> Accelerator -> GPU T4 (or A10G)** -- best quality for 7 B models.
   No GPU? Keep `MODEL_SOURCE = "hf"` and set `MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"`
   (slower, but runs on CPU).
2. **Settings -> toggle "Internet" ON** -- needed to clone the repo and (for the
   `hf` source) download weights. For `kaggle_input` (an attached Kaggle Model)
   you can turn Internet OFF.

Run every cell **top-to-bottom**. Only the **Configuration** cell needs editing.


In [1]:
import sys, subprocess

def _need(pkg, mod):
    try:
        __import__(mod); return False
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return True

for pkg, mod in [
    ("hypothesis", "hypothesis"),
    ("httpx", "httpx"),
    ("python-dotenv", "dotenv"),
    ("transformers", "transformers"),
    ("torch", "torch"),
    ("huggingface_hub", "huggingface_hub"),
    ("kagglehub", "kagglehub"),
]:
    print(("installed" if _need(pkg, mod) else "ok").ljust(9), pkg)

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.9 MB/s eta 0:00:00
installed hypothesis
ok        httpx
ok        python-dotenv
ok        transformers
ok        torch
ok        huggingface_hub
ok        kagglehub
torch 2.10.0+cu128 | cuda: True


### Configuration  (edit THIS cell)

Pick the model source and the fuzzing-loop parameters.

- `MODEL_SOURCE = "hf"` -> download `MODEL_NAME` from the HuggingFace Hub
  (non-gated models work on Kaggle).
- `MODEL_SOURCE = "kaggle_input"` -> read a Kaggle Model you attached via
  **Settings -> Source -> Add model**; paste its local directory (the one
  containing `config.json`) into `KAGGLE_MODEL_REF`, e.g.
  `/kaggle/input/qwen-2-5-7b-instruct`.
- `MODEL_SOURCE = "kaggle_kagglehub"` -> fetch via the `kagglehub` library from
  `KAGGLE_MODEL_REF = "owner/model-slug"` (requires Kaggle credentials).

Sensible defaults are provided; everything else is optional to change.


In [2]:
import os

# Fixed project location (the notebook clones here; keep this path).
PROJECT = "/kaggle/working/Agentic-Fuzzing"
REPO    = "https://github.com/Faseeh24/Agentic-Fuzzing.git"

# ------------------------------------------------------------------
# EDIT THESE VALUES TO CONFIGURE YOUR RUN
# ------------------------------------------------------------------
# Source of the model: "hf" (Hub id) | "kaggle_input" (attached local dir) |
# "kaggle_kagglehub" ("owner/model/version").
MODEL_SOURCE = "hf"

# Source A: HuggingFace Hub repo id. Non-gated models work on Kaggle.
#   "Qwen/Qwen2.5-7B-Instruct"            <- default (best quality, GPU)
#   "mistralai/Mistral-7B-Instruct-v0.3"
#   "Qwen/Qwen2.5-1.5B-Instruct"          <- fast / runs on CPU
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Source B/C: Kaggle Model reference (only used if MODEL_SOURCE != "hf").
#   kaggle_input    -> local dir with config.json, e.g. /kaggle/input/qwen-2-5-7b-instruct
#   kaggle_kagglehub -> "owner/model-slug" or "owner/model-slug/version"
KAGGLE_MODEL_REF = "/kaggle/input/models/qwen-lm/qwen2.5-coder/transformers/14b-instruct/1"

# Fuzzing-loop parameters (the LLM makes one strategy call per iteration)
MAX_ITERATIONS = 5     # seed + refine cycles
NUM_EXAMPLES   = 500   # XML inputs generated & fuzzed per iteration
WALL_CLOCK_CAP = 1800  # seconds, overall safety back-stop
# With a local model there is no per-token cost, so this disables the
# cost-based early-stop gate inside the orchestrator.
COST_BUDGET    = 1e9
# ------------------------------------------------------------------

# Resolve model source -> a value from_pretrained() accepts (Hub id OR local dir).
# The client loads lazily on the first chat() and caches the model singleton.
if MODEL_SOURCE == "hf":
    os.environ["HF_MODEL_NAME"] = MODEL_NAME
elif MODEL_SOURCE == "kaggle_input":
    assert KAGGLE_MODEL_REF, "set KAGGLE_MODEL_REF to the /kaggle/input/... dir"
    os.environ["HF_MODEL_NAME"] = KAGGLE_MODEL_REF
elif MODEL_SOURCE == "kaggle_kagglehub":
    import kagglehub
    os.environ["HF_MODEL_NAME"] = kagglehub.model_download(KAGGLE_MODEL_REF)
else:
    raise SystemExit("MODEL_SOURCE must be 'hf' | 'kaggle_input' | 'kaggle_kagglehub'")

os.environ.setdefault("PYTHONPATH", PROJECT)

print("MODEL SOURCE:", MODEL_SOURCE)
print("MODEL PATH  :", os.environ["HF_MODEL_NAME"])
print("Loop  : iterations=%d  examples/iter=%d  wall_cap=%ds" %
      (MAX_ITERATIONS, NUM_EXAMPLES, int(WALL_CLOCK_CAP)))


MODEL SOURCE: hf
MODEL PATH  : Qwen/Qwen2.5-7B-Instruct
Loop  : iterations=5  examples/iter=500  wall_cap=1800s


### Clone the repo & build the ASan/UBSan C harness

Compiles the vendored Mini-XML library + `harness/mxml_harness.c` with
AddressSanitizer + UndefinedBehaviorSanitizer (same `Makefile` as the repo).


In [3]:
import os, subprocess, shutil

# Clone Agentic-Fuzzing
if not os.path.isdir(os.path.join(PROJECT, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO, PROJECT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT, "pull"], check=False)

os.chdir(PROJECT)
os.environ["PYTHONPATH"] = PROJECT
print("Repo ready at", PROJECT)

# Clone MXML target and checkout required commit
MXML = os.path.join(PROJECT, "target", "mxml")
MXML_COMMIT = "e6824d899d949387fb0156af6f4101373b9be519"

if not os.path.isdir(os.path.join(MXML, ".git")):
    subprocess.run(
        ["git", "clone", "https://github.com/michaelrsweet/mxml.git", MXML],
        check=True
    )

subprocess.run(["git", "-C", MXML, "fetch", "--all"], check=True)
subprocess.run(["git", "-C", MXML, "checkout", MXML_COMMIT], check=True)

# Generate MXML config.h
if not os.path.exists(os.path.join(MXML, "config.h")):
    subprocess.run(["./configure"], cwd=MXML, check=True)

# Build harness
r = subprocess.run(["make", "-C", "harness", "all"], capture_output=True, text=True)
if r.returncode != 0:
    print((r.stderr or "")[-2500:])
    raise SystemExit("Harness build FAILED - see output above.")

assert os.path.exists("harness/mxml_harness"), "mxml_harness binary missing!"
print("Harness built ->", os.path.abspath("harness/mxml_harness"))

Cloning into '/kaggle/working/Agentic-Fuzzing'...


Repo ready at /kaggle/working/Agentic-Fuzzing


Cloning into '/kaggle/working/Agentic-Fuzzing/target/mxml'...


Fetching origin


Note: switching to 'e6824d899d949387fb0156af6f4101373b9be519'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at e6824d8 Use mxml-private.h header in unit test program.


checking build system type... x86_64-pc-linux-gnu
checking host system type... x86_64-pc-linux-gnu
checking for gcc... gcc
checking whether the C compiler works... yes
checking for C compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether the compiler supports GNU C... yes
checking whether gcc accepts -g... yes
checking for gcc option to enable C11 features... none needed
checking for ranlib... ranlib
checking for ar... /usr/bin/ar
checking for codesign... no
checking for true... /usr/bin/true
checking for install-sh script... using /kaggle/working/Agentic-Fuzzing/target/mxml/install-sh
checking for ldconfig... /usr/sbin/ldconfig
checking for mkdir... /usr/bin/mkdir
checking for rm... /usr/bin/rm
checking for rmdir... /usr/bin/rmdir
checking for ln... /usr/bin/ln
checking for stdio.h... yes
checking for stdlib.h... yes
checking for string.h... yes
checking f

### Use an open-source LLM (swap out the Groq client)

The next cell writes a local HuggingFace Transformers `LLMClient` over the
repo's `agent/llm_client.py`. It exposes the *same interface*
(`LLMClient(model=None)`, `.is_available()`, `.chat(messages, timeout)`), so
`agent/orchestrator.py` runs **unchanged**. The model comes from `HF_MODEL_NAME`
(resolved in the Configuration cell above).

Then run the patch cell and the smoke-test cell.


In [4]:
%%writefile /kaggle/working/Agentic-Fuzzing/agent/llm_client.py
"""
agent/llm_client.py — Open-source LLM client for the Kaggle notebook.

This file OVERWRITES the repo's Groq-only client (agent/llm_client.py) so the
same orchestrator pipeline (agent/orchestrator.py) runs locally with any
HuggingFace open-source model instead of calling the Groq HTTP API.

The public interface is identical to the original Groq LLMClient, so
agent/orchestrator.py works UNCHANGED:

    LLMClient(model=None).is_available() -> bool
    LLMClient(model=None).chat(messages, timeout=120.0) -> str

Model selection
---------------
    os.environ["HF_MODEL_NAME"]  -> model repo id (set in the notebook config cell)

The model is loaded lazily on the first chat() call and caches the model singleton.
A module-level singleton cache makes the (slow) load happen only once across every LLMClient() created
during a notebook run — the smoke-test cell pre-loads it, then the pipeline reuses the cached instance.

Robustness
----------
If a model is too big for GPU memory, generation auto-retries on CPU with offloading. If loading
or generation ever fails, chat() raises RuntimeError — the orchestrator catches that exception and
degrades gracefully to the bundled known-good strategy (fuzzer/fallback_strategy.py), so the loop ALWAYS
keeps running.

Suggested (non-gated) models for Kaggle:
    Qwen/Qwen2.5-7B-Instruct          (best quality, needs a GPU T4/A10G)
    mistralai/Mistral-7B-Instruct-v0.3
    Qwen/Qwen2.5-1.5B-Instruct          (fast / works without a GPU)
"""

from __future__ import annotations

import logging
import os
import time
from typing import Optional

import torch

logger = logging.getLogger(__name__)

_DEFAULT_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Singleton cache: model name -> LLMClient instance whose model is loaded.
# Re-using the cache means the heavy download+load happens just once.
_CACHE: dict = {}


def _cuda_available() -> bool:
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False


class LLMClient:
    """Open-source LLM client backed by HuggingFace Transformers."""

    def __init__(self, model: Optional[str] = None) -> None:
        self._model_name = model or os.getenv("HF_MODEL_NAME", _DEFAULT_MODEL)
        self._tokenizer = None
        self._model = None
        self._device = "cuda" if _cuda_available() else "cpu"
        logger.info("LLMClient configured model=%s device=%s",
                     self._model_name, self._device)

    # -- model lifecycle -------------------------------------------------

    def _load(self) -> None:
        """Load (or reuse) the tokenizer + model. Idempotent and cached."""
        cached = _CACHE.get(self._model_name)
        if cached is not None and cached._model is not None:
            self._tokenizer = cached._tokenizer
            self._model = cached._model
            self._device = cached._device
            print(f"[llm_client] Reusing cached model: {self._model_name}")
            return

        from transformers import AutoTokenizer, AutoModelForCausalLM

        print(f"[llm_client] Loading open-source model: {self._model_name}  "
              f"(device={self._device})")
        t0 = time.time()
        self._tokenizer = AutoTokenizer.from_pretrained(self._model_name)
        # Some tokenizers (e.g. Qwen) have no pad token; fall back to eos.
        if self._tokenizer.pad_token is None:
            self._tokenizer.pad_token = self._tokenizer.eos_token

        weight_dtype = torch.bfloat16 if self._device == "cuda" else torch.float32
        try:
            self._model = AutoModelForCausalLM.from_pretrained(
                self._model_name,
                dtype=weight_dtype,
                device_map="auto",
            )
        except Exception as exc:
            # GPU OOM / dtype issues -> retry on CPU with offloading.
            print(f"[llm_client] GPU load failed ({exc!r}); "
                  f"retrying on CPU (float32, offloaded).")
            import gc
            gc.collect()
            try:
                self._model = AutoModelForCausalLM.from_pretrained(
                    self._model_name,
                    dtype=torch.float32,
                    device_map="auto",
                    offload_folder="offload",
                )
            except Exception as exc2:
                raise RuntimeError(
                    f"Failed to load model '{self._model_name}': {exc2!r}"
                ) from exc2

        self._model.eval()
        _CACHE[self._model_name] = self
        print(f"[llm_client] Model '{self._model_name}' loaded "
              f"in {time.time() - t0:.1f}s")

    def is_available(self) -> bool:
        """Return True.

        The local model is treated as available; any load/generation failure
        surfaces as an exception in chat(), which the orchestrator catches and
        degrades to the bundled fallback strategy. This mirrors the Groq client
        whose is_available() gate only guards against a missing API key.
        """
        return True

    # -- generation ------------------------------------------------------

    def chat(self, messages: list[dict], timeout: float = 120.0) -> str:
        """Generate a response from the open-source model.

        Parameters
        ----------
        messages : list[dict]
            OpenAI-style messages, e.g.
            [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}]
        timeout : float
            Accepted for interface compatibility (a local model is not
            rate-limited); not enforced.

        Returns
        -------
        str
            The assistant's response text (everything generated after the prompt).
        """
        if self._model is None:
            self._load()
        if self._model is None or self._tokenizer is None:
            raise RuntimeError(
                f"Open-source model '{self._model_name}' is not loaded."
            )

        # Normalise to plain role/content dicts.
        msgs = [
            {"role": str(m.get("role", "user")), "content": str(m.get("content", ""))}
            for m in messages
        ]

        # Use the tokenizer's native chat template when available
        # (Qwen / Mistral / Llama-3 all ship one). Fall back to a flat concat.
        try:
            input_ids = self._tokenizer.apply_chat_template(
                msgs, return_tensors="pt", add_generation_prompt=True
            )
        except Exception:
            rendered = "\n".join(f"{m['role']}: {m['content']}" for m in msgs)
            enc = self._tokenizer(rendered, return_tensors="pt")
            if hasattr(enc, "input_ids"):
                input_ids = enc.input_ids
            elif isinstance(enc, dict):
                input_ids = enc["input_ids"]
            else:
                input_ids = enc

        # Robustly ensure input_ids is a torch.Tensor
        if not isinstance(input_ids, torch.Tensor):
            if hasattr(input_ids, "input_ids"):
                input_ids = input_ids.input_ids
            elif isinstance(input_ids, list):
                input_ids = torch.tensor(input_ids)
            else:
                try:
                    input_ids = torch.as_tensor(input_ids)
                except Exception as e:
                    raise RuntimeError(f"Failed to convert input_ids to Tensor: {e}")

        input_ids = input_ids.to(self._model.device)

        # Don't let the prompt blow the context window.
        # Use model_max_length if available, otherwise default to 4096.
        max_total = getattr(self._tokenizer, "model_max_length", 4096)
        # Ensure max_total is a reasonable number (sometimes it's inf)
        if not isinstance(max_total, (int, float)) or max_total > 100_000:
            max_total = 4096
        
        tailroom = max_total - 2048
        if input_ids.shape[-1] > tailroom:
            input_ids = input_ids[..., -tailroom:]

        with torch.no_grad():
            generated = self._model.generate(
                input_ids,
                max_new_tokens=4096,
                temperature=0.2,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.05,
                pad_token_id=self._tokenizer.pad_token_id,
                eos_token_id=self._tokenizer.eos_token_id,
            )

        # Drop the prompt and decode only the newly generated tokens.
        prompt_len = input_ids.shape[-1]
        text = self._tokenizer.decode(
            generated[0][prompt_len:], skip_special_tokens=True
        )
        return text


def chat(messages: list[dict], timeout: float = 120.0) -> str:
    """Convenience helper — mirrors the Groq module-level chat()."""
    return LLMClient().chat(messages, timeout=timeout)


Overwriting /kaggle/working/Agentic-Fuzzing/agent/llm_client.py


### Patch the cosmetic provider label

Relabels the `"groq"` provider tag in `agent/orchestrator.py` to `"local-llm"`
so the run summary is accurate. (Orchestrator LOGIC is not touched.)


In [5]:
import os
op = os.path.join(PROJECT, "agent", "orchestrator.py")
src = open(op, encoding="utf-8").read()
patches = [
    ('llm_provider = "groq"', 'llm_provider = "local-llm"'),
    ("mxml (Groq)", "mxml (Local LLM)"),
    ("Groq API key not set. Set GROQ_API_KEY in .env", "open-source model unavailable"),
]
for old, new in patches:
    assert old in src, "pattern not found: " + repr(old)
    src = src.replace(old, new)
open(op, "w", encoding="utf-8").write(src)
print("Patched agent/orchestrator.py -> provider='local-llm'")


Patched agent/orchestrator.py -> provider='local-llm'


### Pre-load the model & smoke test

Downloads (first run only) and loads the model, then asks it for a tiny code
snippet. This gives fast feedback that the LLM backend works before the full
pipeline. The loaded model is cached (`_CACHE`) and reused by the pipeline cell.


In [6]:
import os, sys
os.environ.setdefault("HF_MODEL_NAME", MODEL_NAME)
sys.path.insert(0, PROJECT)

print("Loading model from:", os.environ["HF_MODEL_NAME"])

from agent.llm_client import LLMClient

client = LLMClient()
print("LLM backend:", type(client).__module__ + "." + type(client).__name__,
      "-> model:", client._model_name)
print("is_available:", client.is_available())

resp = client.chat([
    {"role": "system", "content": "You are a terse assistant. Reply with ONLY a python code snippet, no prose."},
    {"role": "user",   "content": "Write a one-line python snippet that prints the number 42."},
], timeout=120)

print("---- model reply ----")
print(resp.strip())
looks_code = "print" in resp.lower() and "42" in resp
if resp.strip() and looks_code:
    print("Smoke test passed - model responds with code.")
else:
    print("WARNING - no code detected; full pipeline will still run (the")
    print("built-in validator + fallback strategy keep things safe).")


Loading model from: Qwen/Qwen2.5-7B-Instruct
LLM backend: agent.llm_client.LLMClient -> model: Qwen/Qwen2.5-7B-Instruct
is_available: True
[llm_client] Loading open-source model: Qwen/Qwen2.5-7B-Instruct  (device=cuda)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[llm_client] Model 'Qwen/Qwen2.5-7B-Instruct' loaded in 97.2s
---- model reply ----
print(42)
Smoke test passed - model responds with code.


### Run the full pipeline

Seeds an XML strategy from the LLM, AST-validates + live-loads it, fuzzes it
through the ASan/UBSan harness, refines using acceptance-rate + crash
signatures, and finally runs crash triage (dedupe / minimize / verify).


In [7]:
import os, sys
os.environ.setdefault("HF_MODEL_NAME", MODEL_NAME)
os.environ["PYTHONPATH"] = PROJECT
sys.path.insert(0, PROJECT)

from agent.orchestrator import run_orchestrator

result = run_orchestrator(
    max_iterations=MAX_ITERATIONS,
    num_examples=NUM_EXAMPLES,
    wall_clock_cap=WALL_CLOCK_CAP,
    cost_budget=COST_BUDGET,
    run_triage=True,
)

print("\n======== PIPELINE RESULT ========")
for k, v in result.items():
    print(f"  {k}: {v}")


Agentic Fuzzing Loop — mxml (Local LLM)

[orch] Generating seed strategy via LLM ...
[llm_client] Reusing cached model: Qwen/Qwen2.5-7B-Instruct
[orch] Seed strategy validated (1446 chars)
[orch] Seed strategy saved → /kaggle/working/Agentic-Fuzzing/fuzzer/strategies/iteration_0000.py

────────────────────────────────────────────────────────────
[orch] Iteration 1
────────────────────────────────────────────────────────────
[orch] Strategy loaded successfully
  provider      : local-llm
  total         : 500  (in 14.9s)
  accept rate   : 25.8%
  valid(0)      : 129
  invalid(1)    : 307
  sanitizer(3)  : 64
  timeout(4)    : 0
  bug_crash(5)  : 0
  crash_count   : 64
  log           : /kaggle/working/Agentic-Fuzzing/fuzzer/logs/iteration_0001.jsonl
  ★ 64 crash candidate(s) found!

  Refining strategy via LLM ...
  Refined strategy saved → /kaggle/working/Agentic-Fuzzing/fuzzer/strategies/iteration_0001.py

────────────────────────────────────────────────────────────
[orch] Iteration 2

No stack trace found for crash; using input-structure fallback. This signature is weaker than a real stack-based one.
No stack trace found for crash; using input-structure fallback. This signature is weaker than a real stack-based one.


  provider      : local-llm
  total         : 500  (in 14.0s)
  accept rate   : 26.6%
  valid(0)      : 133
  invalid(1)    : 301
  sanitizer(3)  : 66
  timeout(4)    : 0
  bug_crash(5)  : 0
  crash_count   : 66
  log           : /kaggle/working/Agentic-Fuzzing/fuzzer/logs/iteration_0005.jsonl
  ★ 66 crash candidate(s) found!

  → Converged; stopping loop.

[orch] Summary written → /kaggle/working/Agentic-Fuzzing/fuzzer/logs/loop_summary.md

Running crash triage ...
Crash Triage Pipeline

[1/4] Collecting crashes from iteration logs ...
  Found 319 crash examples across logs.

[2/4] Deduplicating crashes ...
  319 total -> 4 unique signatures
  Saved 4 crash records to /kaggle/working/Agentic-Fuzzing/triage/crashes

[3/4] Minimizing reproducers ...
  Minimizing 1d30d45458fe6664 (code=3, original_len=64552) ... Test case: _shrink_target(
    candidate='<HF_MODEL_NAME x="b\'\\x00\\xcd R\\x18\\x1aO\\x0c\\xc5\\x87\\xd0P\\xa6\\xb7\\xac\\xd7p\\xca\\xe4\\x96\\xad\\xb9\\xfd\\x8b\\xc7c\\xd7\\xf

/usr/local/lib/python3.12/dist-packages/hypothesis/core.py:1072: HypothesisWarning: Generating overly large repr. This is an expensive operation, and with a length of 78 kB is unlikely to be useful. Use -Wignore to ignore the warning, or -Werror to get a traceback.
  text_repr = repr_call(test, args, kwargs)


Test case: _shrink_target(
    candidate='<V7><addresssanitizer></V7></addresssanitizer>',
)
Traceback (most recent call last):
  File "/kaggle/working/Agentic-Fuzzing/triage/minimize.py", line 292, in _shrink_target
    raise AssertionError(
AssertionError: Still crashes with code 3; input length=46

Test case: _shrink_target(
    candidate='<V7><addresssanitizer></V7></addresssanitizer>',
)
Traceback (most recent call last):
  File "/kaggle/working/Agentic-Fuzzing/triage/minimize.py", line 292, in _shrink_target
    raise AssertionError(
AssertionError: Still crashes with code 3; input length=46

Test case: _shrink_target(
    candidate='<V7><addresssanitizer></V7></addresssanitizer>',
)
Traceback (most recent call last):
  File "/kaggle/working/Agentic-Fuzzing/triage/minimize.py", line 292, in _shrink_target
    raise AssertionError(
AssertionError: Still crashes with code 3; input length=46

Test case: _shrink_target(
    candidate='<V7><addresssanitizer></V7></addresssanitizer>',


...

Traceback (most recent call last):
  File "/kaggle/working/Agentic-Fuzzing/triage/minimize.py", line 292, in _shrink_target
    raise AssertionError(
AssertionError: Still crashes with code 3; input length=60028

minimized_len=60028 (code=3)
  Minimized 4 reproducer(s)

[4/4] Verifying reproducers ...
  Verifying 1d30d45458fe6664 ... 

/usr/local/lib/python3.12/dist-packages/hypothesis/core.py:1072: HypothesisWarning: Generating overly large repr. This is an expensive operation, and with a length of 75 kB is unlikely to be useful. Use -Wignore to ignore the warning, or -Werror to get a traceback.
  text_repr = repr_call(test, args, kwargs)


confirmed
  Verifying 45bedb75c2d7278b ... 

No stack trace found for crash; using input-structure fallback. This signature is weaker than a real stack-based one.
No stack trace found for crash; using input-structure fallback. This signature is weaker than a real stack-based one.
No stack trace found for crash; using input-structure fallback. This signature is weaker than a real stack-based one.


confirmed
  Verifying 50e35c868c105bd9 ... confirmed
  Verifying bff45faa60d47be8 ... confirmed
  4 confirmed, 0 flaky

[orch] Triage complete: 4 unique sigs, 4 confirmed

======== PIPELINE RESULT ========
  state: CRASH_FOUND
  iterations: 5
  total_examples: 2500
  crashes_found: 319
  llm_provider: local-llm
  loop_elapsed: 340.88054180145264
  _log_path: /kaggle/working/Agentic-Fuzzing/fuzzer/logs/loop_summary.md


### Results & artifacts

Inspect generated strategies, per-iteration classification logs, and any crash
reproducers under `triage/crashes/<signature>/`.


In [8]:
import json
from pathlib import Path

root = Path(PROJECT)

summ = root / "fuzzer" / "logs" / "loop_summary.md"
if summ.exists():
    print("loop_summary.md:")
    print(summ.read_text())

print("\nstrategy files:")
for f in sorted((root / "fuzzer/strategies").glob("iteration_*.py")):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

print("\niteration logs:")
for f in sorted((root / "fuzzer/logs").glob("iteration_*.jsonl")):
    rec = json.loads(f.read_text(encoding="utf-8").splitlines()[0])
    r = rec["results"]
    print(f"  {f.name}: total={r['total']} accept={r['acceptance_rate']:.0%} "
          f"sanitizer={r['sanitizer']} timeout={r['timeout']} bug_crash={r['bug_crash']}")

print("\ntriage (crashes):")
cd = root / "triage" / "crashes"
if cd.exists() and any(cd.iterdir()):
    for d in sorted(cd.iterdir()):
        if not d.is_dir():
            continue
        repro = "reproducer_minimized.xml" if (d / "reproducer_minimized.xml").exists() else "reproducer.xml"
        print(f"  {d.name} -> {repro}")
        rep = d / repro
        if rep.exists():
            print("    input:", repr(rep.read_text(encoding="utf-8")[:120]))
        sr = d / "sanitizer_report.txt"
        if sr.exists():
            print("    stderr:", sr.read_text(encoding="utf-8")[:300])
else:
    print("  (no crashes found - nothing to triage)")


loop_summary.md:
# Agentic Loop Summary

**State:** CRASH_FOUND

**Iterations:** 5

**LLM provider:** local-llm

**Wall-clock:** 340.9s

**Total examples:** 2500

**Crashes found:** 319

## Strategy Files

- `iteration_0000.py`
- `iteration_0001.py`
- `iteration_0002.py`
- `iteration_0003.py`
- `iteration_0004.py`

## Log Files

- `iteration_0001.jsonl`
- `iteration_0002.jsonl`
- `iteration_0003.jsonl`
- `iteration_0004.jsonl`
- `iteration_0005.jsonl`


strategy files:
  iteration_0000.py  (1446 bytes)
  iteration_0001.py  (1446 bytes)
  iteration_0002.py  (1446 bytes)
  iteration_0003.py  (1446 bytes)
  iteration_0004.py  (1446 bytes)

iteration logs:
  iteration_0001.jsonl: total=10 accept=20% sanitizer=0 timeout=0 bug_crash=0
  iteration_0002.jsonl: total=500 accept=27% sanitizer=72 timeout=0 bug_crash=0
  iteration_0003.jsonl: total=500 accept=27% sanitizer=56 timeout=0 bug_crash=0
  iteration_0004.jsonl: total=500 accept=25% sanitizer=61 timeout=0 bug_crash=0
  iteration_0005.json

### Save artifacts to Output

Kaggle preserves files under `/kaggle/working/` when you **commit** the run.
To make the generated strategies, per-iteration logs and crash reproducers easy
to recover, this cell copies them into
`/kaggle/working/output/agentic_fuzzing_run/` (shown in the run's Output tab).


In [9]:
import shutil
from pathlib import Path

out_root = Path("/kaggle/working/output/agentic_fuzzing_run")
out_root.mkdir(parents=True, exist_ok=True)
root = Path(PROJECT)

copied = []
for src_sub, dst_name in [
    ("fuzzer/strategies", "strategies"),
    ("fuzzer/logs", "logs"),
    ("triage/crashes", "triage_crashes"),
]:
    src = root / src_sub
    dst = out_root / dst_name
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        copied.append(dst_name)

ls = root / "fuzzer" / "logs" / "loop_summary.md"
if ls.exists():
    shutil.copy2(ls, out_root / "loop_summary.md")

print("Saved artifacts to", out_root)
for name in copied:
    d = out_root / name
    print(" ", name, "/", len(list(d.rglob("*"))), "entries")
for p in sorted(out_root.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(out_root), f"({p.stat().st_size} bytes)")


Saved artifacts to /kaggle/working/output/agentic_fuzzing_run
  strategies / 12 entries
  logs / 6 entries
  triage_crashes / 20 entries
   logs/iteration_0001.jsonl (3617715 bytes)
   logs/iteration_0002.jsonl (3871746 bytes)
   logs/iteration_0003.jsonl (3212479 bytes)
   logs/iteration_0004.jsonl (3295633 bytes)
   logs/iteration_0005.jsonl (3769959 bytes)
   logs/loop_summary.md (439 bytes)
   loop_summary.md (439 bytes)
   strategies/__pycache__/_seed_check.cpython-312.pyc (2875 bytes)
   strategies/__pycache__/_tmp_iteration_0001.cpython-312.pyc (2883 bytes)
   strategies/__pycache__/_tmp_iteration_0002.cpython-312.pyc (2883 bytes)
   strategies/__pycache__/_tmp_iteration_0003.cpython-312.pyc (2883 bytes)
   strategies/__pycache__/_tmp_iteration_0004.cpython-312.pyc (2883 bytes)
   strategies/__pycache__/_tmp_iteration_0005.cpython-312.pyc (2883 bytes)
   strategies/iteration_0000.py (1446 bytes)
   strategies/iteration_0001.py (1446 bytes)
   strategies/iteration_0002.py (1446 b

### Done

- To re-run with a different model: edit the **Configuration** cell (the
  `MODEL_SOURCE` / `MODEL_NAME` / `KAGGLE_MODEL_REF` line), then
  **Runtime -> Restart and run all** (or run from the smoke-test cell onward).
- Generated strategies: `fuzzer/strategies/iteration_*.py`
- Per-example logs (incl. real ASan/UBSan stderr): `fuzzer/logs/iteration_*.jsonl`
- Confirmed crash reproducers: `triage/crashes/<signature>/`
- Archived copy (strategies + logs + crashes + summary) saved to
  `/kaggle/working/output/agentic_fuzzing_run/`
